# Crop Suitability Classification - Capstone Notebook
Run top-to-bottom from the `notebooks/` folder. Mirrors `src/` scripts so every result is reproducible.

In [ ]:
import sys, os; os.chdir('..'); sys.path.insert(0,'src')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from preprocessing import *
pd.set_option('display.width',140)

## 1. Load & inspect

In [ ]:
raw = pd.read_csv('data/crop_data.csv')
print(raw.shape); display(raw.head()); display(raw.describe().round(2))

## 2. Data quality: missing values, duplicates, impossible values

In [ ]:
print('duplicates:', raw.duplicated().sum())
print(raw.isna().sum())
print({c:int(((raw[c]<lo)|(raw[c]>hi)).sum()) for c,(lo,hi) in VALID_RANGES.items()})

In [ ]:
df = clean(raw); print(df.shape); df[TARGET].value_counts().describe()

## 3. EDA

In [ ]:
fig,ax=plt.subplots(2,4,figsize=(16,7))
for a,c in zip(ax.ravel(),FEATURES): sns.histplot(df[c],kde=True,ax=a)
ax.ravel()[-1].axis('off'); plt.show()

In [ ]:
sns.heatmap(df[FEATURES].corr(),annot=True,cmap='coolwarm'); plt.show()

In [ ]:
m=df.groupby(TARGET)[FEATURES].mean(); sns.heatmap((m-m.mean())/m.std(),cmap='RdYlGn',annot=m.round(0),fmt='g'); plt.gcf().set_size_inches(9,8); plt.show()

## 4. Split first (avoid leakage), then model
Imputer + scaler live **inside** the Pipeline so they are fit on training folds only.

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
X,y=df[FEATURES],df[TARGET]
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
cv=StratifiedKFold(5,shuffle=True,random_state=42)
grids={'LogReg':(LogisticRegression(max_iter=2000),{'model__C':[.1,1,10,100]}),
 'KNN':(KNeighborsClassifier(),{'model__n_neighbors':[3,5,7,11,15],'model__weights':['uniform','distance']}),
 'Tree':(DecisionTreeClassifier(random_state=42),{'model__max_depth':[4,6,8,12,None],'model__min_samples_leaf':[1,3,5,10]})}
fit={}
for n,(m_,g) in grids.items():
    gs=GridSearchCV(make_pipeline(m_),g,cv=cv,scoring='f1_macro',n_jobs=-1).fit(Xtr,ytr); fit[n]=gs
    print(n,gs.best_params_,'CV F1=%.3f'%gs.best_score_,'test acc=%.3f'%accuracy_score(yte,gs.predict(Xte)))

## 5. Evaluation of selected model (chosen by CV score, not test score)

In [ ]:
best=max(fit,key=lambda n:fit[n].best_score_); pred=fit[best].predict(Xte)
print(best); print(classification_report(yte,pred))

In [ ]:
labels=sorted(y.unique()); sns.heatmap(confusion_matrix(yte,pred,labels=labels),annot=True,fmt='d',cmap='Greens',xticklabels=labels,yticklabels=labels); plt.gcf().set_size_inches(12,10); plt.show()

## 6. Error analysis
The cell below compares the model's predictions with the true labels on the held-out test set and lists the most frequent confusions between crops (actual → predicted). Crops with similar soil and climate requirements are the most likely to be confused. Per-crop precision, recall and F1-score are in `reports/classification_report.txt`, and the full confusion matrix is in `reports/figures/09_confusion_matrix.png`.

In [ ]:
wrong = Xte.assign(actual=yte, predicted=pred)
wrong = wrong[wrong.actual != wrong.predicted]
print(f"Misclassified: {len(wrong)} of {len(yte)} test samples ({len(wrong)/len(yte):.1%})")
if len(wrong):
    display(wrong.groupby(['actual', 'predicted']).size()
                 .sort_values(ascending=False).head(10)
                 .rename('count').reset_index())
else:
    print("No misclassified samples.")

The full pipeline is saved by `src/train.py` to `models/crop_model.joblib`; the Streamlit app loads it.